In [1]:
import pandas as pd
from scipy import stats
import numpy as np

In [2]:
metrics = pd.read_parquet('../data/processed/person_collaboration_metrics.parquet')
df = pd.read_parquet('../data/processed/iran_cinema_clean.parquet')

In [3]:
metrics.head()

,nconst,primaryName,work_count,unique_collaborators,total_collaborations,hhi,top_collaborator_share,career_length,works_per_active_year
0,nm0000695,Joanne Whalley,1,11,11,0.090909,0.090909,1,1.0
1,nm0001094,Roald Dahl,1,5,5,0.200000,0.200000,1,1.0
2,nm0001120,Vittorio De Sica,1,13,13,0.076923,0.076923,1,1.0
3,nm0001393,Irène Jacob,1,11,11,0.090909,0.090909,1,1.0
4,nm0001411,William Katt,1,11,11,0.090909,0.090909,1,1.0


In [4]:
df_ratings = pd.read_csv('../data/raw/title.ratings.tsv.gz', sep='\t')
df_films = df.drop_duplicates('tconst')

df_films_rated = df_films.merge(df_ratings, on='tconst', how='left')

df_reliable = df_films_rated[df_films_rated['numVotes'] >= 50]  
print(df_reliable.shape)

(1537, 15)


In [5]:
df_1 = df.merge(metrics, on='nconst', how='inner')
df_2 = df_1.merge(df_reliable[['tconst', 'averageRating', 'numVotes']], on='tconst', how='inner')

print(df_2.shape)
# df_2[['primaryName', 'primaryTitle', 'hhi', 'averageRating']].head()
df_2.head()

(16365, 23)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,ordering,...,primaryName_y,work_count,unique_collaborators,total_collaborations,hhi,top_collaborator_share,career_length,works_per_active_year,averageRating,numVotes
0,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",1,...,Nancy Kovack,3,22,30,0.053333,0.100000,2,1.500000,5.0,123.0
1,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",2,...,Reza Fazeli,39,236,371,0.006866,0.026954,48,0.812500,5.0,123.0
2,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",3,...,Taghi Zohuri,72,281,637,0.007056,0.023548,23,3.130435,5.0,123.0
3,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",4,...,Kan'an Kiani,16,135,154,0.008517,0.025974,35,0.457143,5.0,123.0
4,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",5,...,Katayun Amir Ebrahimi,60,365,558,0.004451,0.016129,59,1.016949,5.0,123.0


In [6]:
df_2 = df_2.drop(columns=['primaryName_y']).rename(columns={'primaryName_x': 'primaryName'})

In [7]:
df_2.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,ordering,...,primaryName,work_count,unique_collaborators,total_collaborations,hhi,top_collaborator_share,career_length,works_per_active_year,averageRating,numVotes
0,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",1,...,Nancy Kovack,3,22,30,0.053333,0.100000,2,1.500000,5.0,123.0
1,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",2,...,Reza Fazeli,39,236,371,0.006866,0.026954,48,0.812500,5.0,123.0
2,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",3,...,Taghi Zohuri,72,281,637,0.007056,0.023548,23,3.130435,5.0,123.0
3,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",4,...,Kan'an Kiani,16,135,154,0.008517,0.025974,35,0.457143,5.0,123.0
4,tt0060093,movie,Diamond 33,Almaas 33,0,1967,None,120.0,"Drama,Thriller",5,...,Katayun Amir Ebrahimi,60,365,558,0.004451,0.016129,59,1.016949,5.0,123.0


فرضیه اول: ایا تمرکز همکاری با امتیاز فیلم های حاصل ازهمکاری ارتباط معناداری دارد؟ 

In [8]:
median_hhi = df_2.drop_duplicates('nconst')['hhi'].median()
high_hhi_group = df_2[df_2['hhi'] >= median_hhi]['averageRating']
low_hhi_group = df_2[df_2['hhi'] < median_hhi]['averageRating']

# t-test
result = stats.ttest_ind(high_hhi_group, low_hhi_group, equal_var=False)
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)

# Effect size (Cohen's d)
pooled_sd = np.sqrt((high_hhi_group.var() + low_hhi_group.var()) / 2)
cohens_d = (high_hhi_group.mean() - low_hhi_group.mean()) / pooled_sd
print("Cohen's d:", cohens_d)
print("Mean high HHI:", high_hhi_group.mean(), "| Mean low HHI:", low_hhi_group.mean())

t-statistic: 12.027515726850535
p-value: 6.965328729679121e-33
Cohen's d: 0.2343813449268234
Mean high HHI: 5.367350527549824 | Mean low HHI: 5.046105149386243


فرضیه دوم: آیا فیلم‌هایی که تعداد همکاران متنوع بیشتری دارند، امتیازشان با فیلم‌هایی که همکاران متنوع کمتری دارند، فرق می‌کند؟

In [9]:
median_diversity = df_2.drop_duplicates('nconst')['unique_collaborators'].median()
high_div = df_2[df_2['unique_collaborators'] >= median_diversity]['averageRating']
low_div = df_2[df_2['unique_collaborators'] < median_diversity]['averageRating']

result2 = stats.ttest_ind(high_div, low_div, equal_var=False)
d2 = (high_div.mean() - low_div.mean()) / np.sqrt((high_div.var() + low_div.var())/2)
print("p-value:", result2.pvalue, "| Cohen's d:", d2)

p-value: 2.964705532745157e-36 | Cohen's d: -0.2554483647559975


فرضیه سوم: آیا ژانر خاصی (مثلاً درام در مقابل کمدی) با تمرکز همکاری بیشتری همراهه؟

In [11]:
df_2['is_drama'] = df_2['genres'].str.contains('Drama', na=False)

drama_hhi = df_2[df_2['is_drama']]['hhi']
non_drama_hhi = df_2[~df_2['is_drama']]['hhi']

result3 = stats.ttest_ind(drama_hhi, non_drama_hhi, equal_var=False)
d3 = (drama_hhi.mean() - non_drama_hhi.mean()) / np.sqrt((drama_hhi.var() + non_drama_hhi.var())/2)
print("p-value:", result3.pvalue, "| Cohen's d:", d3)
print("Mean Drama HHI:", drama_hhi.mean(), "| Mean non-Drama HHI:", non_drama_hhi.mean())

p-value: 8.431368295995039e-09 | Cohen's d: -0.1054576910389485
Mean Drama HHI: 0.029551788684566326 | Mean non-Drama HHI: 0.037760906074697585


In [12]:
from statsmodels.stats.multitest import multipletests
p_values = [6.965328729679121e-33, result2.pvalue, 8.431368295995039e-09]
reject, adjusted_p, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
print(adjusted_p, reject)

[1.04479931e-32 8.89411660e-36 8.43136830e-09] [ True  True  True]
